# Step 1: Give the agent one Python tool

This lesson adds one small tool that only reads a fixed list. The model can decide to call it, AgentScope runs it, and the result helps the model write its final answer.

In [ ]:
import os

from dotenv import load_dotenv

from agentscope.agent import Agent, ReActConfig
from agentscope.credential import OpenAICredential
from agentscope.message import Msg, TextBlock
from agentscope.model import OpenAIChatModel
from agentscope.tool import FunctionTool, Toolkit

## Step 2: Define the practice IP-details tool

You are a junior analyst on a practice security team at a company. One employee's work computer contacted several internet addresses, and you must prepare a short note. The tool below checks a small practice list for details about an IP address (an internet address). Its results are clues, not proof by themselves.

| Local result | Meaning in this course |
| --- | --- |
| `known benign` | Expected activity in the practice list; not a promise that the address is always safe. |
| `suspicious` | A person should look into it; not proof that anything bad happened. |
| `no record` | The practice list has no entry; that does not mean the address is safe. |

In [ ]:
def get_ip_details(ip_address: str) -> dict[str, str]:
    """Get the practice-list details for one internet address.

    Inputs:
        ip_address: An internet address, such as 192.0.2.44, supplied by the agent.

    Output:
        A practice-list record, or a result saying no record was found.

    Process:
        1. Use the input IP address as a key in the fixed records dictionary.
        2. If a key matches, return that stored record unchanged.
        3. Otherwise, construct and return a no-record dictionary that
           preserves the supplied IP address.
    """
    # The fixed list gives the same answer each time and needs no internet connection.
    records = {
        "198.51.100.10": {
            "ip_address": "198.51.100.10",
            "record_found": "yes",
            "local_result": "known benign",
            "details": "This address is an expected software-update mirror in the training scenario.",
            "source": "local practice list",
        },
        "192.0.2.44": {
            "ip_address": "192.0.2.44",
            "record_found": "yes",
            "local_result": "suspicious",
            "details": "This address is marked for analyst follow-up in the training scenario.",
            "source": "local practice list",
        },
    }
    return records.get(
        ip_address,
        {
            "ip_address": ip_address,
            "record_found": "no",
            "local_result": "no record",
            "details": "No details are available in the local practice list.",
            "source": "local practice list",
        },
    )


# FunctionTool turns this regular Python function into a tool the agent can use.
# The model sees the tool description, not this function or its fixed list.
# This tool only reads fixed in-memory data, so it may run without approval.
ip_details_tool = FunctionTool(get_ip_details, is_read_only=True)

# Toolkit is the agent's toolbox: it stores tools the agent may use.
# This lesson's toolbox contains only get_ip_details.
toolkit = Toolkit(tools=[ip_details_tool])

# Inspect the tool description that AgentScope sends to the model.
tool_schemas = await toolkit.get_tool_schemas()
tool_schemas

## Step 3: Configure the model and create the tool-using agent

The next cell loads the local model settings and creates an agent with the toolkit from the previous cell, so the agent can request the IP-details tool.

In [ ]:
load_dotenv()

model_name = os.getenv("MODEL")
base_url = os.getenv("OLLAMA_BASE_URL")
if not model_name or not base_url:
    raise RuntimeError("Set MODEL and OLLAMA_BASE_URL in .env before running this notebook.")

model = OpenAIChatModel(
    credential=OpenAICredential(api_key="ollama", base_url=base_url),
    model=model_name,
    stream=False,
    parameters=OpenAIChatModel.Parameters(temperature=0, max_tokens=220),
)

triage_agent = Agent(
    name="triage_assistant",
    system_prompt=(
        "You are a careful assistant helping review computer activity. "
        "For every IP-address question, call get_ip_details before answering. "
        "Use only the tool result when describing what is known about an IP address."
    ),
    model=model,
    # Give the agent its toolbox. Without this line, it cannot call get_ip_details.
    toolkit=toolkit,
    react_config=ReActConfig(max_iters=3),
)

## Step 4: Review tool inputs and outputs

| Input to `get_ip_details` | Returned output |
| --- | --- |
| `"198.51.100.10"` | A practice-list record marked `known benign`. |
| `"192.0.2.44"` | A practice-list record marked `suspicious`. |
| `"198.51.100.23"` | A result that says no local record was found. |

The example is important: the function keeps the requested address but does not make up details when the practice list has no match.

## Step 5: Run the ReAct loop

![AgentScope ReAct workflow with one Python tool](figures/python-tool-react-workflow.svg)

The model first receives the user message and a description of the tool. When it asks for the tool, AgentScope runs the matching function and adds the result to the conversation. The model then sees that result and writes its final answer. The limit of three tries is a safety limit, not a request for three calls.

In [ ]:
# reply() runs the loop: model → tool call → function result → model → final text.
response = await triage_agent.reply(
    Msg(
        name="analyst",
        role="user",
        content=[
            TextBlock(
                text=(
                    "What details does the local dataset have about 192.0.2.44? "
                    "Call the tool first, then answer in two sentences."
                ),
            ),
        ],
    ),
)

print("".join(block.text for block in response.content if isinstance(block, TextBlock)))